<a href="https://colab.research.google.com/github/Elnazzareei/E-Commerce-Sales-Analysis-SQL-Data-Pipeline-Business-Insights/blob/main/E-Commerce-Sales-Analysis-SQL-Data-Pipeline-Business-Insights.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Load data
from google.colab import files
uploaded = files.upload()

Saving ecommerce_sales_dataset.csv to ecommerce_sales_dataset.csv


In [2]:
# Explor data
import pandas as pd
df=pd.read_csv('ecommerce_sales_dataset.csv')
print(df.head())
print(df.shape)
print(df.dtypes)

     Order_ID  Order_Date  Year  Month Quarter  Season Customer_ID  \
0  ORD-000001   1/18/2023  2023      1      Q1  Winter  CUST-02791   
1  ORD-000002  11/13/2023  2023     11      Q4    Fall  CUST-02763   
2  ORD-000003   4/28/2022  2022      4      Q2  Spring  CUST-05729   
3  ORD-000004  10/22/2021  2021     10      Q4    Fall  CUST-06490   
4  ORD-000005    8/6/2023  2023      8      Q3  Summer  CUST-07594   

  Customer_Gender Customer_Segment         Region  ... Discount  Revenue  \
0            Male              New  North America  ...     0.00  3642.80   
1          Female          Regular  North America  ...     0.25    33.88   
2            Male          Regular  North America  ...     0.00   241.30   
3          Female          Regular           Asia  ...     0.00   654.50   
4          Female          Premium         Europe  ...     0.20   178.82   

      Cost   Profit  Profit_Margin_%  Shipping_Cost  Shipping_Method  \
0  2444.11  1198.69            32.91           2.2

In [3]:
# Convert data (flat table to SQL relational databse)
Customers=df[[
    'Customer_ID',
    'Customer_Gender',
    'Customer_Segment',
    'Country',
    'Region'
]].drop_duplicates(subset=['Customer_ID']).copy()


Orders=df[[
    'Order_ID',
    'Customer_ID',
    'Order_Date',
    'Year',
    'Month',
    'Quarter',
    'Season',
    'Payment_Method',
    'Order_Status'
]].drop_duplicates(subset=['Order_ID']).copy()

Products=df[[
    'Product_Name',
    'Category',
    'Sub_Category',
    'Unit_Price'
]].drop_duplicates().reset_index(drop=True)
Products['Products_ID']=Products.index+1
df=df.merge(Products,on=['Product_Name','Category','Sub_Category','Unit_Price'])

Order_items=df[[
    'Order_ID',
    'Products_ID',
    'Quantity',
    'Discount',
    'Revenue',
    'Cost',
    'Profit',
    'Profit_Margin_%'
]].copy()
Order_items['Order_items_ID']=range(1,len(Order_items)+1)

Shipping=df[[
    'Order_ID',
    'Shipping_Cost',
    'Shipping_Method',
    'Shipping_Days'
]].drop_duplicates(subset=['Order_ID']).copy()
Shipping['Shipping_ID']=Shipping.index+1

In [24]:
# Load tables into SQLite
import sqlite3
conn=sqlite3.connect('ecommerce_sql_project.db')
cursor=conn.cursor()

# Create SQL tabels
cursor.execute("""
CREATE TABLE IF NOT EXISTS Customers(
  Customer_id TEXT PRIMARY KEY,
  Customer_Gender TEXT,
  Customer_Segment TEXT,
  Country TEXT,
  Region TEXT
  );
  """)

cursor.execute("""
CREATE TABLE IF NOT EXISTS Orders(
  Order_ID TEXT PRIMARY KEY,
  Customer_ID TEXT,
  Order_Date TEXT,
  Year INTEGER,
  Month INTEGER,
  Quarter TEXT,
  Season TEXT,
  Payment_Method TEXT,
  Order_Status TEXT
  );
  """)

cursor.execute("""
CREATE TABLE IF NOT EXISTS Products(
  Products_ID INTEGER PRIMARY KEY,
  Product_Name TEXT,
  Category TEXT,
  Sub_Category TEXT,
  Unit_Price REAL
  );
  """)

cursor.execute("""
CREATE TABLE IF NOT EXISTS Order_items(
  Order_items_ID INTEGER PRIMARY KEY,
  Order_ID TEXT,
  Products_ID INTEGER,
  Quantity INTEGER,
  Discount REAL,
  Revenue REAL,
  Cost REAL,
  Profit REAL,
  Profit_Margin_ REAL,
  FOREIGN KEY (Order_ID) REFERENCES Orders(Order_ID),
  FOREIGN KEY (Products_ID) REFERENCES Products(Products_ID)
  );
  """)

cursor.execute("""
CREATE TABLE IF NOT EXISTS Shipping(
  Shipping_ID INTEGER PRIMARY KEY,
  Order_ID TEXT,
  Shipping_Cost REAL,
  Shipping_Method TEXT,
  Shipping_Days INTEGER,
  FOREIGN KEY (Order_ID) REFERENCES Orders(Order_ID)
  );
  """)

conn.commit()


# Insert data from pandas
Customers.to_sql('Customers',conn,if_exists='replace', index=False)
Orders.to_sql('Orders',conn,if_exists='replace', index=False)
Products.to_sql('Products', conn, if_exists='replace', index=False)
Order_items.to_sql('Order_items', conn, if_exists='replace', index=False)
Shipping.to_sql('Shipping', conn, if_exists='replace', index=False)

# Verify everything
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print(cursor.fetchall())

tables=['Customers', 'Orders', 'Products', 'Order_items', 'Shipping']
for table in tables:
  cursor.execute(f"SELECT COUNT(*) from {table}")
  print(table, cursor.fetchone())


# Check for duplicate Order_IDs
print(Orders['Order_ID'].duplicated().sum())
print(Shipping['Order_ID'].duplicated().sum())
print(Customers['Customer_ID'].duplicated().sum())
print(Products['Products_ID'].duplicated().sum())
print(Order_items['Order_items_ID'].duplicated().sum())

[('Customers',), ('Orders',), ('Products',), ('Order_items',), ('Shipping',)]
Customers (5348,)
Orders (10000,)
Products (9916,)
Order_items (10000,)
Shipping (10000,)
0
0
0
0
0


In [7]:
# SQL Analysis Phase
# Revenue by Country
query="""
SELECT Country, ROUND(SUM(Revenue),2) as Total_Revenue
FROM Order_items
JOIN Orders ON Order_items.Order_ID=Orders.Order_ID
JOIN Customers ON Orders.Customer_ID=Customers.Customer_ID
GROUP BY Country
ORDER BY Total_Revenue DESC;
"""
pd.read_sql_query(query,conn)

,Country,Total_Revenue
0,USA,483595.97
1,Mexico,475921.47
2,Canada,472244.92
3,India,316416.67
4,Japan,312973.82
5,Italy,308122.41
6,Egypt,292846.26
7,Jordan,284521.71
8,China,283081.30
9,Spain,253402.86


In [9]:
# Revenue by category
query="""
SELECT Category, ROUND(SUM(Revenue),2) as Total_Revenue
FROM Order_items
JOIN Orders ON Order_items.Order_ID=Orders.Order_ID
JOIN Products ON Order_items.Products_ID=Products.Products_ID
GROUP BY Category
ORDER BY Total_Revenue DESC;
"""
pd.read_sql_query(query,conn)

,Category,Total_Revenue
0,Electronics,3382028.46
1,Home & Kitchen,892842.02
2,Clothing,407471.79
3,Books & Media,386644.46
4,Beauty & Health,215400.97


In [10]:
# Top products by revenue
query="""
SELECT Products_ID, ROUND(SUM(Revenue),2) as Total_Revenue
FROM Order_items
GROUP BY Products_ID
ORDER BY Total_Revenue DESC
LIMIT 10;
"""
pd.read_sql_query(query,conn)

,Products_ID,Total_Revenue
0,266,19966.50
1,604,19246.08
2,8040,19017.04
3,2984,16901.36
4,8946,16056.69
5,2117,15142.47
6,8337,14855.47
7,6552,14010.25
8,9289,13398.93
9,4124,13366.14


In [11]:
# Top customers
query="""
SELECT Customers.Customer_ID, Country, ROUND(SUM(Revenue),2) as Total_Spent
FROM Customers Customers
JOIN Orders ON Customers.Customer_ID=Orders.Customer_ID
JOIN Order_items ON Orders.Order_ID=Order_items.Order_ID
GROUP BY Customers.Customer_ID
ORDER BY Total_Spent DESC
LIMIT 10;
"""
pd.read_sql_query(query,conn)

,Customer_ID,Country,Total_Spent
0,CUST-01046,Japan,20094.80
1,CUST-02471,UK,20064.82
2,CUST-06024,Canada,19700.13
3,CUST-04179,Mexico,16901.36
4,CUST-03717,Spain,16567.88
5,CUST-05654,Jordan,15482.04
6,CUST-04026,France,14898.21
7,CUST-05700,Saudi Arabia,14067.20
8,CUST-07378,South Korea,13464.60
9,CUST-03852,Egypt,13398.93


In [12]:
# Profit by category
query="""
SELECT Category, ROUND(SUM(Profit),2) as Total_Profit
FROM Order_items
JOIN Orders ON Orders.Order_ID=Order_items.Order_ID
JOIN Products ON Order_items.Products_ID=Products.Products_ID
GROUP BY Category
ORDER BY Total_Profit DESC;
"""
pd.read_sql_query(query,conn)

,Category,Total_Profit
0,Electronics,925448.15
1,Home & Kitchen,232421.45
2,Clothing,115705.29
3,Books & Media,105037.69
4,Beauty & Health,59025.73


In [13]:
# Monthly revenue trend
query="""
SELECT Month, Year, ROUND(SUM(Revenue),2) as Monthly_Revenue
FROM Orders
JOIN Order_items ON Orders.Order_ID=Order_items.Order_ID
GROUP BY Month, Year
ORDER BY Month, Year;
"""
pd.read_sql_query(query,conn)

,Month,Year,Monthly_Revenue
0,1,2021,9276.43
1,1,2022,99447.76
2,1,2023,160492.14
3,1,2024,216708.74
4,2,2021,8321.14
5,2,2022,80586.86
6,2,2023,137356.66
7,2,2024,130107.66
8,3,2021,7377.79
9,3,2022,81247.66


In [14]:
# Window function
query="""
SELECT Country, SUM(Revenue) as Total_Revenue_Country,
RANK() OVER (ORDER BY SUM(Revenue) DESC) as Country_Rank
FROM Customers
JOIN Orders ON Customers.Customer_ID=Orders.Customer_ID
JOIN Order_items ON Orders.Order_ID=Order_items.Order_ID
GROUP BY Country;
"""
pd.read_sql_query(query,conn)

,Country,Total_Revenue_Country,Country_Rank
0,USA,483595.97,1
1,Mexico,475921.47,2
2,Canada,472244.92,3
3,India,316416.67,4
4,Japan,312973.82,5
5,Italy,308122.41,6
6,Egypt,292846.26,7
7,Jordan,284521.71,8
8,China,283081.30,9
9,Spain,253402.86,10


In [15]:
# Export SQLite database file
import os
os.listdir()
Customers.to_csv('Customers.csv', index=False)
Orders.to_csv('Orders.csv', index=False)
Products.to_csv('Products.csv', index=False)
Order_items.to_csv('Order_items.csv', index=False)
Shipping.to_csv('Shipping.csv', index=False)

In [16]:
os.listdir()
from google.colab import files

In [17]:
# Download files
files.download('Customers.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
files.download('Orders.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
files.download('Products.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
files.download('Order_items.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
files.download('Shipping.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
# Close connection
conn.close()